# Notebook 3: Generative AI Concepts for Physicists

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

This notebook builds intuition for:
- What generative models do (and don't do)
- How **diffusion models** work — with physics analogies
- Why diffusion models are well-suited for crystal structure generation
- How **structural constraints** fit into the generation process

> This is mostly a conceptual notebook with visualizations. Minimal coding required.

In [ ]:
# --- Run once per Colab session (skip if you already ran 00_setup.ipynb) ---
# This notebook only needs numpy, matplotlib, scipy (pre-installed on Colab).
# Install scipy just in case:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scipy'])
print('Ready.')

---
## 3.1 Discriminative vs. generative models

| | Discriminative | Generative |
|---|---|---|
| **Task** | Predict a label from data | Create new data samples |
| **Learns** | P(label \| data) | P(data) |
| **Materials example** | "Is this structure stable?" | "Generate a new stable structure" |

A generative model learns the **probability distribution** over its training data,
then samples from that distribution to create new examples.

**Physics analogy:** Think of the training data as defining an energy landscape.
The generative model learns this landscape, and sampling is like finding new
low-energy configurations.

---
## 3.2 A brief taxonomy of generative models

| Model | Key idea | Physics analogy |
|-------|----------|-----------------|
| **VAE** | Encode → latent space → decode | Compressed representation of configuration space |
| **GAN** | Generator vs. discriminator game | Adversarial optimization |
| **Flow** | Invertible transformations | Canonical transformations in phase space |
| **Diffusion** | Gradually add then remove noise | Heating → controlled cooling (annealing) |

**Diffusion models** have become the dominant approach for crystal generation
because they naturally handle the periodicity and symmetry of crystal structures.

---
## 3.3 How diffusion models work

### The forward process: adding noise

Start with a real crystal structure and gradually corrupt it:
1. Displace atoms from their positions (add Gaussian noise to coordinates)
2. Randomize atom types
3. Distort the lattice parameters

After enough steps, the structure becomes pure noise.

**Physics analogy:** This is like **heating a crystal until it melts** into a
disordered liquid. Each step adds a tiny bit of thermal energy.

### The reverse process: denoising

A neural network learns to **reverse** this corruption — to denoise step by step.
Starting from random noise, it gradually builds a valid crystal structure.

**Physics analogy:** This is like **controlled crystallization from a melt**.
The model has learned the "crystallization dynamics" and can guide random atoms
into a coherent crystal.

### What does the network learn?

Technically, the network learns the **score function** ∇log p(x) — the gradient
of the log-probability. In plain terms: "which direction should I move each atom
to make this look more like a real crystal?"

Let's visualize this with a simple 2D example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

np.random.seed(42)

# Define a simple 2D target distribution (mixture of Gaussians arranged in a ring)
def target_samples(n=500):
    """Generate samples from a ring of Gaussians (like crystal sites)."""
    n_centers = 6
    angles = np.linspace(0, 2*np.pi, n_centers, endpoint=False)
    centers = np.stack([2*np.cos(angles), 2*np.sin(angles)], axis=1)
    samples = []
    for _ in range(n):
        c = centers[np.random.randint(n_centers)]
        samples.append(c + 0.15 * np.random.randn(2))
    return np.array(samples)

# Simulate the forward (noising) process
data = target_samples(500)
noise_levels = [0.0, 0.3, 0.8, 2.0]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, sigma in zip(axes, noise_levels):
    noised = data + sigma * np.random.randn(*data.shape)
    ax.scatter(noised[:, 0], noised[:, 1], s=8, alpha=0.5, c='steelblue')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    if sigma == 0:
        ax.set_title('Original data\n(crystal sites)', fontsize=11)
    else:
        ax.set_title(f'Noise level σ = {sigma}', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('Forward process: gradually adding noise ("heating")', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Simulate the reverse (denoising) process
# Starting from noise, gradually denoise toward the target distribution

def simple_denoise_step(x, data, sigma_current, sigma_next, n_neighbors=10):
    """A toy denoising step: move points toward nearest cluster centers."""
    from scipy.spatial.distance import cdist
    # Estimate score: direction toward nearest data points
    dists = cdist(x, data)
    nearest_idx = np.argsort(dists, axis=1)[:, :n_neighbors]
    target = np.mean(data[nearest_idx], axis=1)
    # Move partway toward target
    step_size = 1 - sigma_next / sigma_current
    x_new = x + step_size * (target - x)
    # Add small noise (except at final step)
    if sigma_next > 0:
        x_new += sigma_next * 0.3 * np.random.randn(*x.shape)
    return x_new

# Start from pure noise
x = 3.5 * np.random.randn(500, 2)
sigmas = [2.0, 0.8, 0.3, 0.0]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].scatter(x[:, 0], x[:, 1], s=8, alpha=0.5, c='coral')
axes[0].set_title('Pure noise\n("disordered melt")', fontsize=11)
axes[0].set_xlim(-5, 5); axes[0].set_ylim(-5, 5)
axes[0].set_aspect('equal'); axes[0].set_xticks([]); axes[0].set_yticks([])

for i in range(1, 4):
    x = simple_denoise_step(x, data, sigmas[i-1], sigmas[i] if i < 3 else 0)
    color = ['coral', 'mediumpurple', 'steelblue'][i-1]
    axes[i].scatter(x[:, 0], x[:, 1], s=8, alpha=0.5, c=color)
    if i < 3:
        axes[i].set_title(f'Denoising step {i}', fontsize=11)
    else:
        axes[i].set_title('Generated samples\n("crystallized")', fontsize=11)
    axes[i].set_xlim(-5, 5); axes[i].set_ylim(-5, 5)
    axes[i].set_aspect('equal'); axes[i].set_xticks([]); axes[i].set_yticks([])

fig.suptitle('Reverse process: denoising ("controlled crystallization")', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

The toy example above shows the core idea:
- The **forward process** destroys structure by adding noise
- The **reverse process** creates structure by removing noise
- The neural network learns to perform the reverse process

In a real crystal diffusion model (like SCIGEN), the same principle applies
but in 3D, with periodic boundary conditions, and the "data" includes:
- Atom positions (fractional coordinates)
- Atom types (element species)
- Lattice parameters (the unit cell shape)

---
## 3.5 Conditional generation and structural constraints

A diffusion model can generate **unconditionally** (any valid crystal) or
**conditionally** (crystals with specific properties):

| Conditioning type | Example | Method |
|---|---|---|
| Property-guided | "Band gap > 2 eV" | Classifier guidance |
| Composition-guided | "Must contain Mn and O" | Fix atom types during denoising |
| **Structure-guided** | **"Must have kagome sublattice"** | **Inpainting with constrained atoms** |

SCIGEN's key innovation is **structure-guided generation** via an **inpainting** approach:
you specify a geometric constraint (kagome, honeycomb, etc.), and the model generates
complete crystal structures that contain that sublattice.

### How does it work?

The approach is analogous to **image inpainting** — filling in missing parts of a picture:

1. The model generates the entire structure through its normal denoising process
2. At the end, the "known" atom positions (defining the kagome/honeycomb/etc. pattern) are **revealed** — replacing whatever the model generated at those sites
3. The model learns to generate structures that are **consistent** with these constraints, because during training it sees the constrained positions and learns to build coherent crystals around them
4. The lattice shape is also constrained (e.g., hexagonal cell for kagome)

This is different from simply clamping atoms in place at every denoising step.
Instead, the model learns to **anticipate** the constraint and generate complementary
atoms that form a physically reasonable crystal around the fixed sublattice.

# Visualize the inpainting idea: constrained kagome sites + generated additional atoms
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Kagome constraint points
a = 5.0
gamma = np.radians(120)
lat_a = np.array([a, 0])
lat_b = np.array([a * np.cos(gamma), a * np.sin(gamma)])

kagome_frac = np.array([[0.5, 0.0], [0.0, 0.5], [0.5, 0.5]])

def frac_to_cart(frac, lat_a, lat_b):
    return frac[:, 0:1] * lat_a + frac[:, 1:2] * lat_b

# Plot 1: The desired constraint pattern
ax = axes[0]
for di in range(-1, 3):
    for dj in range(-1, 3):
        shifted = kagome_frac + np.array([[di, dj]])
        cart = frac_to_cart(shifted, lat_a, lat_b)
        ax.scatter(cart[:, 0], cart[:, 1], s=100, c='orchid',
                   edgecolors='black', linewidths=0.8, zorder=5)
ax.set_xlim(-2, 12); ax.set_ylim(-2, 11)
ax.set_aspect('equal')
ax.set_title('Target: kagome sublattice\n(will be "inpainted")', fontsize=11)

# Plot 2: Model generates from noise (all atoms are noisy)
ax = axes[1]
np.random.seed(7)
all_noisy = np.random.uniform(-1, 11, (45, 2))
ax.scatter(all_noisy[:, 0], all_noisy[:, 1], s=60, c='lightgray',
           edgecolors='gray', linewidths=0.5, alpha=0.7, zorder=4)
ax.set_xlim(-2, 12); ax.set_ylim(-2, 11)
ax.set_aspect('equal')
ax.set_title('Denoising: model generates\nall atoms from noise', fontsize=11)

# Plot 3: After denoising, kagome sites are revealed (inpainted)
ax = axes[2]
for di in range(-1, 3):
    for dj in range(-1, 3):
        shifted = kagome_frac + np.array([[di, dj]])
        cart = frac_to_cart(shifted, lat_a, lat_b)
        ax.scatter(cart[:, 0], cart[:, 1], s=100, c='orchid',
                   edgecolors='black', linewidths=0.8, zorder=5)

# Simulated generated additional atoms at sensible positions
extra_frac = np.array([[0.33, 0.33], [0.17, 0.17], [0.67, 0.33], [0.33, 0.67]])
for di in range(-1, 3):
    for dj in range(-1, 3):
        shifted = extra_frac + np.array([[di, dj]])
        cart = frac_to_cart(shifted, lat_a, lat_b)
        ax.scatter(cart[:, 0], cart[:, 1], s=60, c='steelblue',
                   edgecolors='black', linewidths=0.5, zorder=4)
ax.set_xlim(-2, 12); ax.set_ylim(-2, 11)
ax.set_aspect('equal')
ax.set_title('Result: kagome sites inpainted\n+ generated atoms (blue)', fontsize=11)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
# Visualize the idea: fixed kagome sites + generated additional atoms
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Kagome constraint points
a = 5.0
gamma = np.radians(120)
lat_a = np.array([a, 0])
lat_b = np.array([a * np.cos(gamma), a * np.sin(gamma)])

kagome_frac = np.array([[0.5, 0.0], [0.0, 0.5], [0.5, 0.5]])

def frac_to_cart(frac, lat_a, lat_b):
    return frac[:, 0:1] * lat_a + frac[:, 1:2] * lat_b

# Plot 1: Just the constraint
ax = axes[0]
for di in range(-1, 3):
    for dj in range(-1, 3):
        shifted = kagome_frac + np.array([[di, dj]])
        cart = frac_to_cart(shifted, lat_a, lat_b)
        ax.scatter(cart[:, 0], cart[:, 1], s=100, c='orchid',
                   edgecolors='black', linewidths=0.8, zorder=5)
ax.set_xlim(-2, 12); ax.set_ylim(-2, 11)
ax.set_aspect('equal')
ax.set_title('Step 1: Fixed kagome sites\n(structural constraint)', fontsize=11)

# Plot 2: Constraint + noise for unknown atoms
ax = axes[1]
for di in range(-1, 3):
    for dj in range(-1, 3):
        shifted = kagome_frac + np.array([[di, dj]])
        cart = frac_to_cart(shifted, lat_a, lat_b)
        ax.scatter(cart[:, 0], cart[:, 1], s=100, c='orchid',
                   edgecolors='black', linewidths=0.8, zorder=5)
np.random.seed(7)
noisy_atoms = np.random.uniform(-1, 11, (30, 2))
ax.scatter(noisy_atoms[:, 0], noisy_atoms[:, 1], s=60, c='lightgray',
           edgecolors='gray', linewidths=0.5, alpha=0.7, zorder=4)
ax.set_xlim(-2, 12); ax.set_ylim(-2, 11)
ax.set_aspect('equal')
ax.set_title('Step 2: Add noisy atoms\n(start of denoising)', fontsize=11)

# Plot 3: Constraint + denoised atoms
ax = axes[2]
for di in range(-1, 3):
    for dj in range(-1, 3):
        shifted = kagome_frac + np.array([[di, dj]])
        cart = frac_to_cart(shifted, lat_a, lat_b)
        ax.scatter(cart[:, 0], cart[:, 1], s=100, c='orchid',
                   edgecolors='black', linewidths=0.8, zorder=5)

# Simulated "denoised" additional atoms at sensible positions
extra_frac = np.array([[0.33, 0.33], [0.17, 0.17], [0.67, 0.33], [0.33, 0.67]])
for di in range(-1, 3):
    for dj in range(-1, 3):
        shifted = extra_frac + np.array([[di, dj]])
        cart = frac_to_cart(shifted, lat_a, lat_b)
        ax.scatter(cart[:, 0], cart[:, 1], s=60, c='steelblue',
                   edgecolors='black', linewidths=0.5, zorder=4)
ax.set_xlim(-2, 12); ax.set_ylim(-2, 11)
ax.set_aspect('equal')
ax.set_title('Step 3: Generated structure\n(kagome + additional atoms)', fontsize=11)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

---
## 3.6 The landscape of crystal generative models

| Model | Year | Type | Key feature |
|-------|------|------|-------------|
| CDVAE | 2022 | VAE | First crystal generative model with graph decoder |
| DiffCSP | 2023 | Diffusion | Crystal structure prediction from composition |
| **SCIGEN** | **2025** | **Diffusion** | **Structural constraints (kagome, honeycomb, etc.)** |
| MatterGen | 2025 | Diffusion | Property-guided generation |
| UniMat | 2025 | Diffusion | Universal materials generation |

What makes SCIGEN unique: it is the first model that lets you specify **geometric
constraints** on the generated structure. Instead of just saying "generate something
stable," you can say "generate a structure with a kagome sublattice" — and the model
will fill in the rest of the crystal around that constraint.

---
## 3.7 Available structural constraints in SCIGEN

SCIGEN supports 13 lattice types. Here are the most commonly used:

In [ ]:
# Available structural constraints in SCIGEN
sc_table = {
    'tri':  ('Triangular',              1, '60°'),
    'hon':  ('Honeycomb',               2, '120°'),
    'kag':  ('Kagome',                  3, '120°'),
    'sqr':  ('Square',                  1, '90°'),
    'lieb': ('Lieb',                    3, '90°'),
    'elt':  ('Elongated triangular',    2, 'variable'),
    'sns':  ('Snub square',             4, 'variable'),
    'tsq':  ('Truncated square',        4, '90°'),
    'srt':  ('Small rhombotrihexagonal', 6, '120°'),
    'snh':  ('Snub hexagonal',          6, '120°'),
    'trh':  ('Truncated hexagonal',     6, '120°'),
    'grt':  ('Great rhombotrihexagonal', 12, '120°'),
    'van':  ('Vanilla (no constraint)',  1, 'any'),
}

print(f'{"Code":<6} {"Lattice type":<30} {"Known atoms":<14} {"γ angle"}')
print('-' * 60)
for code, (name, n_known, gamma) in sc_table.items():
    print(f'{code:<6} {name:<30} {n_known:<14} {gamma}')

---
## Key takeaways

1. **Diffusion models** generate crystals by learning to reverse a noising process — like controlled crystallization from a melt.
2. The model learns a **score function**: "which direction to move atoms to make the structure more crystal-like."
3. **SCIGEN** adds structural constraints: fix some atoms in a desired geometric pattern, let the model generate the rest.
4. This enables **targeted discovery** of materials with specific lattice geometries (kagome magnets, honeycomb materials, etc.).

In the next notebook, we'll learn how to **evaluate** whether generated structures are physically reasonable.

---
## What's next?

Proceed to **Notebook 04: Evaluating Generated Materials** to learn about MLIPs and structure screening.